# Kaggle - Train ClinVar Window-Attention CNN From Raw Files

Notebook standalone cho Kaggle: bắt đầu từ `variant_summary.txt` và `ncbi_dataset.zip`, giải nén FASTA GRCh38, lọc ClinVar SNV như notebook 01, build tensor `ref+alt+marker`, rồi train/evaluate `dilated_window_attention` CNN.

Notebook này không import code từ `scripts/` và không sửa notebook 01.

In [ ]:
# =========================
# 1. Config
# =========================
from pathlib import Path

CONTEXT_LENGTH = 2001  # đổi được: 601, 1001, 2001, 4001
assert CONTEXT_LENGTH % 2 == 1, "CONTEXT_LENGTH must be odd so the variant has a center index"
CONTEXT_FLANK = CONTEXT_LENGTH // 2

RUN_DEMO = True
RUN_FULL_BUILD = True
RUN_FULL_TRAIN = True
FORCE_REBUILD_DATA = False
FORCE_RETRAIN = False

# Kaggle working disk is limited. Keep this False to avoid writing a huge dense X_ref_alt_marker_*.npy.
CACHE_DENSE_TENSOR = False

SPLIT_MODES = ["gene_group", "purged_gene_group"]
PURGE_DISTANCE_BP = 5000

RANDOM_STATE = 42
BATCH_SIZE = 128
EPOCHS = 5
HARD_NEGATIVE_EPOCHS = 2
LEARNING_RATE = 1e-3
HARD_NEGATIVE_LR = 3e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 1.0
WARMUP_EPOCHS = 1.0
MIN_LR_RATIO = 0.05
POSITIONAL_ENCODING = "sinusoidal"
POSITIONAL_DIM = 8
RC_AUGMENT = True
RC_TTA = True
NUM_WORKERS = 0

DEBUG_MAX_TRAIN_ROWS = 2048
DEBUG_MAX_VAL_ROWS = 512
DEBUG_MAX_TEST_ROWS = 512
DEBUG_BATCH_SIZE = 32

IS_KAGGLE = Path("/kaggle/input").exists()
WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd() / "processed" / "kaggle_work"
PROCESSED_DIR = WORK_DIR / "processed"
MODEL_DIR = WORK_DIR / "models"
UNZIP_DIR = WORK_DIR / "ncbi_dataset_unzipped"
for path in [WORK_DIR, PROCESSED_DIR, MODEL_DIR, UNZIP_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("WORK_DIR:", WORK_DIR)
print("CONTEXT_LENGTH:", CONTEXT_LENGTH)
print("SPLIT_MODES:", SPLIT_MODES)

In [ ]:
# =========================
# 2. Imports and packages
# =========================
import json
import math
import os
import random
import shutil
import subprocess
import sys
import time
import zipfile
from collections import Counter

import numpy as np
import pandas as pd

try:
    from pyfaidx import Fasta
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyfaidx"])
    from pyfaidx import Fasta

try:
    import pyarrow  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyarrow"])

import torch
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 80)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_STATE)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
# =========================
# 3. Locate raw files and FASTA (supports extracted Kaggle datasets or zip)
# =========================
def first_existing(paths):
    for path in paths:
        if path.exists():
            return path
    return None

search_roots = []
if Path("/kaggle/input").exists():
    search_roots.append(Path("/kaggle/input"))
search_roots.extend([Path.cwd(), Path.cwd() / "Data", Path.cwd().parent / "Data"])
search_roots = [p for p in search_roots if p.exists()]

variant_candidates = []
zip_candidates = []
fasta_candidates = []
for root in search_roots:
    variant_candidates.extend(root.rglob("variant_summary.txt"))
    zip_candidates.extend(root.rglob("ncbi_dataset.zip"))
    fasta_candidates.extend(root.rglob("*GRCh38*genomic.fna"))
    fasta_candidates.extend(root.rglob("*GRCh38*genomic.fa"))
    fasta_candidates.extend(root.rglob("*GRCh38*genomic.fasta"))

VARIANT_SUMMARY_PATH = first_existing(variant_candidates)
NCBI_ZIP_PATH = first_existing(zip_candidates)
FASTA_PATH = first_existing(fasta_candidates)

if VARIANT_SUMMARY_PATH is None:
    raise FileNotFoundError("Could not find variant_summary.txt under Kaggle input or local Data/")

print("variant_summary:", VARIANT_SUMMARY_PATH)
if FASTA_PATH is not None:
    print("Using already extracted FASTA:", FASTA_PATH)
elif NCBI_ZIP_PATH is not None:
    print("ncbi_dataset.zip:", NCBI_ZIP_PATH)
    with zipfile.ZipFile(NCBI_ZIP_PATH) as zf:
        fasta_members = [name for name in zf.namelist() if name.endswith((".fna", ".fa", ".fasta"))]
        grch38_members = [name for name in fasta_members if "GRCh38" in name and "genomic" in name]
        print("FASTA members:", len(fasta_members))
        print("GRCh38 candidates:", grch38_members[:5])

    if not any(UNZIP_DIR.rglob("*GRCh38*genomic.fna")):
        print("Extracting ncbi_dataset.zip to", UNZIP_DIR)
        with zipfile.ZipFile(NCBI_ZIP_PATH) as zf:
            zf.extractall(UNZIP_DIR)
    else:
        print("Using existing extracted FASTA in", UNZIP_DIR)

    extracted_fasta = list(UNZIP_DIR.rglob("*GRCh38*genomic.fna"))
    if not extracted_fasta:
        extracted_fasta = [p for p in UNZIP_DIR.rglob("*.fna") if "GRCh38" in p.name]
    FASTA_PATH = first_existing(extracted_fasta)
else:
    raise FileNotFoundError(
        "Could not find either extracted GRCh38 FASTA or ncbi_dataset.zip. "
        "On Kaggle, attach the dataset containing variant_summary.txt and the extracted ncbi_dataset folder or the zip."
    )

if FASTA_PATH is None or not FASTA_PATH.exists():
    raise FileNotFoundError("Could not find GRCh38 FASTA after search/unzip")

print("FASTA_PATH:", FASTA_PATH)

In [ ]:
# =========================
# 4. Demo raw read: first chunk only
# =========================
RAW_USECOLS = [
    "Type",
    "GeneSymbol",
    "ClinicalSignificance",
    "Assembly",
    "ChromosomeAccession",
    "Chromosome",
    "ReviewStatus",
    "PositionVCF",
    "ReferenceAlleleVCF",
    "AlternateAlleleVCF",
]

first_chunk = pd.read_csv(
    VARIANT_SUMMARY_PATH,
    sep="\t",
    usecols=RAW_USECOLS,
    chunksize=100_000,
    low_memory=False,
).__next__()
print(first_chunk.shape)
display(first_chunk.head())
print("Type counts in first chunk:")
display(first_chunk["Type"].value_counts(dropna=False).head(10))
print("ClinicalSignificance counts in first chunk:")
display(first_chunk["ClinicalSignificance"].value_counts(dropna=False).head(10))

In [ ]:
# =========================
# 5. Full chunked ClinVar filtering
# =========================
POSITIVE_LABELS = {"Pathogenic", "Likely pathogenic", "Pathogenic/Likely pathogenic"}
NEGATIVE_LABELS = {"Benign", "Likely benign", "Benign/Likely benign"}
LABEL_MAP = {label: 1 for label in POSITIVE_LABELS}
LABEL_MAP.update({label: 0 for label in NEGATIVE_LABELS})
TRUSTED_REVIEW_PATTERN = "criteria provided|reviewed by expert panel|practice guideline"
LOW_CONFIDENCE_REVIEW = "no assertion criteria provided"
ACGT = set("ACGT")

CHROM_TO_REFSEQ = {
    "1": "NC_000001.11",
    "2": "NC_000002.12",
    "3": "NC_000003.12",
    "4": "NC_000004.12",
    "5": "NC_000005.10",
    "6": "NC_000006.12",
    "7": "NC_000007.14",
    "8": "NC_000008.11",
    "9": "NC_000009.12",
    "10": "NC_000010.11",
    "11": "NC_000011.10",
    "12": "NC_000012.12",
    "13": "NC_000013.11",
    "14": "NC_000014.9",
    "15": "NC_000015.10",
    "16": "NC_000016.10",
    "17": "NC_000017.11",
    "18": "NC_000018.10",
    "19": "NC_000019.10",
    "20": "NC_000020.11",
    "21": "NC_000021.9",
    "22": "NC_000022.11",
    "X": "NC_000023.11",
    "Y": "NC_000024.10",
    "MT": "NC_012920.1",
    "M": "NC_012920.1",
}

BASE_TO_INDEX = np.full(256, 255, dtype=np.uint8)
for _idx, _base in enumerate(b"ACGT"):
    BASE_TO_INDEX[_base] = _idx


def clean_base(value):
    return str(value).strip().upper()


def map_clean_label(value):
    if pd.isna(value):
        return np.nan
    return LABEL_MAP.get(str(value).strip(), np.nan)


def normalize_chromosome(value):
    if pd.isna(value):
        return ""
    text = str(value).strip()
    if text.lower().startswith("chr"):
        text = text[3:]
    return text.upper()


def filter_clinvar_chunks(force=False):
    output_path = PROCESSED_DIR / "clinvar_kaggle_filtered_candidates.parquet"
    if output_path.exists() and not force:
        print("Using existing filtered candidates:", output_path)
        return pd.read_parquet(output_path)

    kept_chunks = []
    counters = Counter()
    reader = pd.read_csv(
        VARIANT_SUMMARY_PATH,
        sep="\t",
        usecols=RAW_USECOLS,
        chunksize=500_000,
        low_memory=False,
    )
    for chunk in tqdm(reader, desc="Filter ClinVar chunks"):
        counters["raw_rows"] += len(chunk)
        chunk = chunk.copy()
        chunk["ClinicalSignificance"] = chunk["ClinicalSignificance"].astype(str).str.strip()
        chunk["Y"] = chunk["ClinicalSignificance"].map(map_clean_label)
        mask = chunk["Y"].notna()
        counters["labeled_rows"] += int(mask.sum())

        mask &= chunk["Assembly"].eq("GRCh38")
        counters["grch38_labeled_rows"] += int(mask.sum())
        mask &= chunk["Type"].eq("single nucleotide variant")
        counters["snv_rows"] += int(mask.sum())

        review = chunk["ReviewStatus"].fillna("").astype(str)
        trusted = review.str.contains(TRUSTED_REVIEW_PATTERN, case=False, regex=True)
        low_conf = review.str.fullmatch(LOW_CONFIDENCE_REVIEW, case=False)
        mask &= trusted & ~low_conf
        counters["trusted_rows"] += int(mask.sum())

        ref = chunk["ReferenceAlleleVCF"].map(clean_base)
        alt = chunk["AlternateAlleleVCF"].map(clean_base)
        mask &= ref.str.fullmatch("[ACGT]") & alt.str.fullmatch("[ACGT]")
        pos = pd.to_numeric(chunk["PositionVCF"], errors="coerce")
        mask &= pos.notna()

        out = chunk.loc[mask].copy()
        if len(out) == 0:
            continue
        out["ReferenceAlleleVCF"] = ref.loc[mask].to_numpy()
        out["AlternateAlleleVCF"] = alt.loc[mask].to_numpy()
        out["PositionVCF"] = pos.loc[mask].astype(np.int64).to_numpy()
        out["Y"] = out["Y"].astype(np.int8)
        out["Chromosome"] = out["Chromosome"].map(normalize_chromosome)
        kept_chunks.append(out)

    if not kept_chunks:
        raise RuntimeError("Filtering produced no rows")
    filtered = pd.concat(kept_chunks, ignore_index=True)
    filtered.to_parquet(output_path, index=False)
    print("Counters:", dict(counters))
    print("Filtered candidates:", filtered.shape)
    display(filtered["Y"].value_counts().sort_index())
    display(filtered["ReviewStatus"].value_counts().head(10))
    return filtered

candidate_df = filter_clinvar_chunks(force=FORCE_REBUILD_DATA)
display(candidate_df.head())

In [ ]:
# =========================
# 6. FASTA join demo and REF check on sample
# =========================
genome = Fasta(str(FASTA_PATH), as_raw=True, sequence_always_upper=True)
fasta_names = set(genome.keys())
print("FASTA sequences:", len(fasta_names))
print(list(sorted(fasta_names))[:5])


def add_chromosome_fasta(df):
    out = df.copy()
    accession = out["ChromosomeAccession"].fillna("").astype(str).str.strip()
    chrom = out["Chromosome"].map(normalize_chromosome)
    fallback = chrom.map(CHROM_TO_REFSEQ).fillna("")
    out["ChromosomeFASTA"] = np.where(accession.isin(fasta_names), accession, fallback)
    return out


def get_reference_base(row):
    chrom = row["ChromosomeFASTA"]
    pos = int(row["PositionVCF"])
    if chrom not in genome or pos < 1:
        return None
    return str(genome[chrom][pos - 1:pos]).upper()

candidate_df = add_chromosome_fasta(candidate_df)
candidate_df = candidate_df[candidate_df["ChromosomeFASTA"].isin(fasta_names)].reset_index(drop=True)
print("Rows with ChromosomeFASTA in FASTA:", len(candidate_df))

demo_fasta_df = candidate_df.head(20).copy()
demo_fasta_df["ReferenceBaseGRCh38"] = demo_fasta_df.apply(get_reference_base, axis=1)
demo_fasta_df["ReferenceAlleleMatchesGRCh38"] = demo_fasta_df["ReferenceBaseGRCh38"].eq(demo_fasta_df["ReferenceAlleleVCF"])
display(demo_fasta_df[["Chromosome", "ChromosomeAccession", "ChromosomeFASTA", "PositionVCF", "ReferenceAlleleVCF", "ReferenceBaseGRCh38", "ReferenceAlleleMatchesGRCh38"]])
assert demo_fasta_df["ReferenceAlleleMatchesGRCh38"].any(), "No REF match in the demo sample"

In [ ]:
# =========================
# 7. Build metadata/y and optional dense tensor
# =========================
def encode_sequence(sequence):
    encoded = BASE_TO_INDEX[np.frombuffer(sequence.encode("ascii"), dtype=np.uint8)]
    if np.any(encoded == 255):
        raise ValueError("sequence contains non-ACGT bases")
    return encoded


def extract_context_metadata(df, force=False):
    metadata_path = PROCESSED_DIR / f"clinvar_training_metadata_{CONTEXT_LENGTH}.parquet"
    y_path = PROCESSED_DIR / f"y_{CONTEXT_LENGTH}.npy"
    context_path = PROCESSED_DIR / f"clinvar_context_{CONTEXT_LENGTH}.parquet"
    if metadata_path.exists() and y_path.exists() and not force:
        print("Using existing metadata:", metadata_path)
        metadata = pd.read_parquet(metadata_path)
        y = np.load(y_path, mmap_mode="r")
        return metadata, y

    rows = []
    skip_counts = Counter()
    for row in tqdm(df.itertuples(index=False), total=len(df), desc=f"Extract {CONTEXT_LENGTH} bp metadata"):
        chrom = str(row.ChromosomeFASTA)
        if chrom not in genome:
            skip_counts["chrom_missing"] += 1
            continue
        pos = int(row.PositionVCF)
        start_1based = pos - CONTEXT_FLANK
        end_1based = pos + CONTEXT_FLANK
        if start_1based < 1:
            skip_counts["edge_or_short"] += 1
            continue
        seq = str(genome[chrom][start_1based - 1:end_1based])
        if len(seq) != CONTEXT_LENGTH:
            skip_counts["edge_or_short"] += 1
            continue
        ref = str(row.ReferenceAlleleVCF)
        if seq[CONTEXT_FLANK] != ref:
            skip_counts["ref_mismatch"] += 1
            continue
        try:
            encode_sequence(seq)
        except ValueError:
            skip_counts["non_acgt_context"] += 1
            continue
        record = row._asdict()
        record["ReferenceBaseGRCh38"] = seq[CONTEXT_FLANK]
        record["ReferenceAlleleMatchesGRCh38"] = True
        record["ContextStart1Based"] = start_1based
        record["ContextEnd1Based"] = end_1based
        rows.append(record)

    metadata = pd.DataFrame(rows)
    y = metadata["Y"].to_numpy(dtype=np.int8)
    metadata.to_parquet(metadata_path, index=False)
    np.save(y_path, y)
    context_cols = ["Chromosome", "PositionVCF", "ReferenceAlleleVCF", "AlternateAlleleVCF", "ChromosomeFASTA", "ContextStart1Based", "ContextEnd1Based", "Y"]
    metadata[context_cols].to_parquet(context_path, index=False)
    print("Metadata rows:", len(metadata))
    print("Skip counts:", dict(skip_counts))
    print("Label counts:", dict(zip(*np.unique(y, return_counts=True))))
    return metadata, y


def write_context_tensor(metadata, force=False):
    x_path = PROCESSED_DIR / f"X_ref_alt_marker_{CONTEXT_LENGTH}.npy"
    if x_path.exists() and not force:
        print("Using existing tensor:", x_path)
        return np.load(x_path, mmap_mode="r")
    n_rows = len(metadata)
    positions = np.arange(CONTEXT_LENGTH)
    x = np.lib.format.open_memmap(x_path, mode="w+", dtype=np.uint8, shape=(n_rows, CONTEXT_LENGTH, 9))
    for i, row in enumerate(tqdm(metadata.itertuples(index=False), total=n_rows, desc="Write one-hot tensor")):
        start = int(row.ContextStart1Based)
        end = int(row.ContextEnd1Based)
        seq = str(genome[str(row.ChromosomeFASTA)][start - 1:end])
        ref_encoded = encode_sequence(seq)
        alt_encoded = ref_encoded.copy()
        alt_encoded[CONTEXT_FLANK] = BASE_TO_INDEX[ord(str(row.AlternateAlleleVCF))]
        x[i, :, :] = 0
        x[i, POSITIONS, ref_encoded] = 1
        x[i, POSITIONS, alt_encoded + 4] = 1
        x[i, CONTEXT_FLANK, 8] = 1
    x.flush()
    return np.load(x_path, mmap_mode="r")


POSITIONS = np.arange(CONTEXT_LENGTH)

if RUN_FULL_BUILD:
    metadata_df, y = extract_context_metadata(candidate_df, force=FORCE_REBUILD_DATA)
else:
    metadata_df = pd.read_parquet(PROCESSED_DIR / f"clinvar_training_metadata_{CONTEXT_LENGTH}.parquet")
    y = np.load(PROCESSED_DIR / f"y_{CONTEXT_LENGTH}.npy", mmap_mode="r")

if CACHE_DENSE_TENSOR:
    X = write_context_tensor(metadata_df, force=FORCE_REBUILD_DATA)
else:
    X = None
    estimated_dense_gb = len(metadata_df) * CONTEXT_LENGTH * 9 / 1024**3
    print(f"CACHE_DENSE_TENSOR=False: skip dense tensor cache. Estimated dense tensor would be {estimated_dense_gb:.2f} GiB")

print("X:", None if X is None else (X.shape, X.dtype))
print("y:", y.shape, y.dtype)
print("metadata:", metadata_df.shape)

In [ ]:
# =========================
# 8. Demo tensor/on-the-fly encoding checks
# =========================
def build_ref_alt_marker_from_metadata_index(idx):
    row = metadata_df.iloc[int(idx)]
    start = int(row.ContextStart1Based)
    end = int(row.ContextEnd1Based)
    seq = str(genome[str(row.ChromosomeFASTA)][start - 1:end])
    if len(seq) != CONTEXT_LENGTH:
        raise ValueError(f"Bad sequence length for row {idx}: {len(seq)}")
    ref_encoded = encode_sequence(seq)
    alt_encoded = ref_encoded.copy()
    alt_encoded[CONTEXT_FLANK] = BASE_TO_INDEX[ord(str(row.AlternateAlleleVCF))]
    x = np.zeros((CONTEXT_LENGTH, 9), dtype=np.float32)
    x[POSITIONS, ref_encoded] = 1.0
    x[POSITIONS, alt_encoded + 4] = 1.0
    x[CONTEXT_FLANK, 8] = 1.0
    return x


def get_encoded_sample(idx):
    if X is not None:
        return X[int(idx)].astype(np.float32, copy=True)
    return build_ref_alt_marker_from_metadata_index(int(idx))

if RUN_DEMO:
    sample_n = min(20, len(metadata_df))
    sample_x = np.stack([get_encoded_sample(i) for i in range(sample_n)])
    assert sample_x.shape == (sample_n, CONTEXT_LENGTH, 9)
    assert int(sample_x[:, CONTEXT_FLANK, 8].sum()) == sample_n
    base_names = np.array(list("ACGT"))
    ref_center_idx = sample_x[:, CONTEXT_FLANK, 0:4].argmax(axis=1)
    alt_center_idx = sample_x[:, CONTEXT_FLANK, 4:8].argmax(axis=1)
    ref_center = base_names[ref_center_idx]
    alt_center = base_names[alt_center_idx]
    check = metadata_df.head(sample_n).copy()
    check["tensor_ref_center"] = ref_center
    check["tensor_alt_center"] = alt_center
    check["ref_ok"] = check["tensor_ref_center"].eq(check["ReferenceAlleleVCF"])
    check["alt_ok"] = check["tensor_alt_center"].eq(check["AlternateAlleleVCF"])
    display(check[["ReferenceAlleleVCF", "AlternateAlleleVCF", "tensor_ref_center", "tensor_alt_center", "ref_ok", "alt_ok"]])
    assert check["ref_ok"].all()
    assert check["alt_ok"].all()

In [ ]:
# =========================
# 9. Split functions and audits
# =========================
def make_groups(metadata):
    groups = metadata["GeneSymbol"].fillna("unknown").astype(str).to_numpy(copy=True)
    unknown = (groups == "") | (groups == "unknown") | (groups == "nan")
    row_ids = np.arange(len(groups)).astype(str)
    groups[unknown] = np.char.add("unknown_row_", row_ids[unknown])
    return groups


def make_gene_group_split(y_array, groups, random_state):
    all_idx = np.arange(len(y_array))
    gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=random_state)
    train_idx, temp_idx = next(gss1.split(all_idx, y_array, groups=groups))
    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=random_state)
    val_rel, test_rel = next(gss2.split(temp_idx, y_array[temp_idx], groups=groups[temp_idx]))
    return train_idx, temp_idx[val_rel], temp_idx[test_rel]


def nearest_distance_to_reference(metadata, query_idx, reference_idx):
    pos_by_chr = {}
    for chrom, sub in metadata.iloc[reference_idx].groupby("Chromosome", sort=False):
        pos_by_chr[str(chrom)] = np.sort(pd.to_numeric(sub["PositionVCF"], errors="coerce").dropna().to_numpy(dtype=np.int64))
    query = metadata.iloc[query_idx]
    query_chr = query["Chromosome"].astype(str).to_numpy()
    query_pos = pd.to_numeric(query["PositionVCF"], errors="coerce").to_numpy()
    distances = np.full(len(query_idx), np.inf, dtype=np.float64)
    for i, (chrom, pos) in enumerate(zip(query_chr, query_pos)):
        if pd.isna(pos):
            continue
        arr = pos_by_chr.get(str(chrom))
        if arr is None or len(arr) == 0:
            continue
        pos_int = int(pos)
        j = np.searchsorted(arr, pos_int)
        best = np.inf
        if j > 0:
            best = min(best, abs(pos_int - int(arr[j - 1])))
        if j < len(arr):
            best = min(best, abs(pos_int - int(arr[j])))
        distances[i] = best
    return distances


def distance_summary(distances):
    finite = distances[np.isfinite(distances)]
    if len(finite) == 0:
        return {"min": float("inf"), "p5": float("inf"), "median": float("inf"), "p95": float("inf")}
    return {"min": float(np.min(finite)), "p5": float(np.percentile(finite, 5)), "median": float(np.median(finite)), "p95": float(np.percentile(finite, 95))}


def maybe_subsample(indices, max_rows, y_array, seed):
    if max_rows is None or len(indices) <= max_rows:
        return indices
    rng = np.random.default_rng(seed)
    selected = []
    for label in [0, 1]:
        label_idx = indices[y_array[indices] == label]
        n_label = int(round(max_rows * len(label_idx) / len(indices)))
        n_label = min(len(label_idx), max(1, n_label))
        selected.append(rng.choice(label_idx, size=n_label, replace=False))
    out = np.concatenate(selected)
    rng.shuffle(out)
    return out


def prepare_split(split_mode, debug=False):
    groups = make_groups(metadata_df)
    train_idx, val_idx, test_idx = make_gene_group_split(y, groups, RANDOM_STATE)
    purged_val_idx = np.array([], dtype=np.int64)
    purged_test_idx = np.array([], dtype=np.int64)
    if split_mode == "purged_gene_group":
        val_dist = nearest_distance_to_reference(metadata_df, val_idx, train_idx)
        keep_val = val_dist >= PURGE_DISTANCE_BP
        purged_val_idx = val_idx[~keep_val]
        val_idx = val_idx[keep_val]
        test_ref = np.concatenate([train_idx, val_idx])
        test_dist = nearest_distance_to_reference(metadata_df, test_idx, test_ref)
        keep_test = test_dist >= PURGE_DISTANCE_BP
        purged_test_idx = test_idx[~keep_test]
        test_idx = test_idx[keep_test]
    if debug:
        train_idx = maybe_subsample(train_idx, DEBUG_MAX_TRAIN_ROWS, y, RANDOM_STATE)
        val_idx = maybe_subsample(val_idx, DEBUG_MAX_VAL_ROWS, y, RANDOM_STATE + 1)
        test_idx = maybe_subsample(test_idx, DEBUG_MAX_TEST_ROWS, y, RANDOM_STATE + 2)
    audit = make_split_audit(split_mode, groups, train_idx, val_idx, test_idx)
    return train_idx, val_idx, test_idx, purged_val_idx, purged_test_idx, audit


def label_counts(indices):
    labels, counts = np.unique(y[indices], return_counts=True)
    return {int(k): int(v) for k, v in zip(labels, counts)}


def make_split_audit(split_mode, groups, train_idx, val_idx, test_idx):
    train_genes = set(groups[train_idx])
    val_genes = set(groups[val_idx])
    test_genes = set(groups[test_idx])
    val_dist = nearest_distance_to_reference(metadata_df, val_idx, train_idx)
    test_dist = nearest_distance_to_reference(metadata_df, test_idx, np.concatenate([train_idx, val_idx]))
    audit = {
        "split_mode": split_mode,
        "train_rows": int(len(train_idx)),
        "val_rows": int(len(val_idx)),
        "test_rows": int(len(test_idx)),
        "train_label_counts": label_counts(train_idx),
        "val_label_counts": label_counts(val_idx),
        "test_label_counts": label_counts(test_idx),
        "gene_overlap_train_val": int(len(train_genes & val_genes)),
        "gene_overlap_train_test": int(len(train_genes & test_genes)),
        "gene_overlap_val_test": int(len(val_genes & test_genes)),
        "val_nearest_train_distance_bp": distance_summary(val_dist),
        "test_nearest_train_val_distance_bp": distance_summary(test_dist),
    }
    if split_mode == "purged_gene_group":
        assert audit["val_nearest_train_distance_bp"]["min"] >= PURGE_DISTANCE_BP
        assert audit["test_nearest_train_val_distance_bp"]["min"] >= PURGE_DISTANCE_BP
    return audit

for split_mode in SPLIT_MODES:
    _, _, _, _, _, audit = prepare_split(split_mode, debug=True)
    print(split_mode)
    print(json.dumps(audit, indent=2))

In [ ]:
# =========================
# 10. Dataset, RC augmentation/TTA, positional encoding
# =========================
def make_sinusoidal_position_encoding(length, dim):
    if dim == 0:
        return np.zeros((length, 0), dtype=np.float32)
    if dim % 2 != 0:
        raise ValueError("POSITIONAL_DIM must be even")
    positions = np.arange(length, dtype=np.float32)[:, None]
    div_term = np.exp(np.arange(0, dim, 2, dtype=np.float32) * (-np.log(10000.0) / dim))
    pe = np.zeros((length, dim), dtype=np.float32)
    pe[:, 0::2] = np.sin(positions * div_term)
    pe[:, 1::2] = np.cos(positions * div_term)
    return pe


def make_relative_position_encoding(length, dim):
    if dim == 0:
        return np.zeros((length, 0), dtype=np.float32)
    if dim % 2 != 0:
        raise ValueError("POSITIONAL_DIM must be even")
    center = (length - 1) / 2.0
    positions = (np.arange(length, dtype=np.float32) - center)[:, None]
    div_term = np.exp(np.arange(0, dim, 2, dtype=np.float32) * (-np.log(10000.0) / dim))
    pe = np.zeros((length, dim), dtype=np.float32)
    pe[:, 0::2] = np.sin(positions * div_term)
    pe[:, 1::2] = np.cos(positions * div_term)
    return pe


def make_position_encoding(kind, length, dim):
    if kind == "none":
        return np.zeros((length, 0), dtype=np.float32)
    if kind == "sinusoidal":
        return make_sinusoidal_position_encoding(length, dim)
    if kind == "relative":
        return make_relative_position_encoding(length, dim)
    raise ValueError(kind)


def reverse_complement_ref_alt_marker(x):
    rc = x[::-1].copy()
    rc[:, 0:4] = rc[:, [3, 2, 1, 0]]
    rc[:, 4:8] = rc[:, [7, 6, 5, 4]]
    return rc


class ClinVarSequenceDataset(Dataset):
    def __init__(self, x_array, y_array, indices, positional_encoding, random_reverse_complement=False, reverse_complement=False):
        self.x_array = x_array
        self.y_array = y_array
        self.indices = np.asarray(indices, dtype=np.int64)
        self.positional_encoding = positional_encoding.astype(np.float32, copy=False)
        self.random_reverse_complement = random_reverse_complement
        self.reverse_complement = reverse_complement
        if self.random_reverse_complement and self.reverse_complement:
            raise ValueError("Use either random or deterministic reverse-complement")

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        idx = int(self.indices[item])
        if self.x_array is None:
            x = build_ref_alt_marker_from_metadata_index(idx)
        else:
            x = self.x_array[idx].astype(np.float32, copy=True)
        if self.reverse_complement or (self.random_reverse_complement and random.random() < 0.5):
            x = reverse_complement_ref_alt_marker(x)
        x = np.concatenate([x, self.positional_encoding], axis=1)
        x = np.ascontiguousarray(x.T)
        return torch.from_numpy(x), torch.tensor(np.float32(self.y_array[idx]))


def make_weighted_sampler(indices, y_array):
    labels = y_array[indices].astype(int)
    counts = np.bincount(labels, minlength=2)
    weights_by_class = 1.0 / np.sqrt(np.maximum(counts, 1))
    sample_weights = weights_by_class[labels]
    return WeightedRandomSampler(torch.as_tensor(sample_weights, dtype=torch.double), num_samples=len(sample_weights), replacement=True)


def make_loader(indices, positional_encoding, batch_size, sampler=None, shuffle=False, random_reverse_complement=False, reverse_complement=False):
    dataset = ClinVarSequenceDataset(X, y, indices, positional_encoding, random_reverse_complement=random_reverse_complement, reverse_complement=reverse_complement)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle if sampler is None else False, sampler=sampler, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

positional_encoding = make_position_encoding(POSITIONAL_ENCODING, CONTEXT_LENGTH, POSITIONAL_DIM)
INPUT_CHANNELS = 9 + positional_encoding.shape[1]
print("INPUT_CHANNELS:", INPUT_CHANNELS)
print("Dense tensor cache:", X is not None)

In [ ]:
# =========================
# 11. Dilated CNN + 1D window attention model
# =========================
class WindowAttentionBlock1D(nn.Module):
    def __init__(self, channels, window_size=51, num_heads=4, dropout=0.10):
        super().__init__()
        if channels % num_heads != 0:
            raise ValueError("channels must be divisible by num_heads")
        self.window_size = window_size
        self.left_pad = window_size // 2
        self.norm_attn = nn.LayerNorm(channels)
        self.attn = nn.MultiheadAttention(channels, num_heads, dropout=dropout, batch_first=True)
        self.norm_ffn = nn.LayerNorm(channels)
        self.ffn = nn.Sequential(
            nn.Linear(channels, channels * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(channels * 2, channels),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        batch_size, channels, length = x.shape
        x_seq = x.transpose(1, 2)
        right_pad = (-length - self.left_pad) % self.window_size
        if self.left_pad or right_pad:
            x_seq = F.pad(x_seq, (0, 0, self.left_pad, right_pad))
        padded_length = x_seq.shape[1]
        windows = x_seq.reshape(batch_size, padded_length // self.window_size, self.window_size, channels)
        windows = windows.reshape(-1, self.window_size, channels)
        attn_input = self.norm_attn(windows)
        attn_output, _ = self.attn(attn_input, attn_input, attn_input, need_weights=False)
        windows = windows + attn_output
        windows = windows + self.ffn(self.norm_ffn(windows))
        x_seq = windows.reshape(batch_size, padded_length // self.window_size, self.window_size, channels)
        x_seq = x_seq.reshape(batch_size, padded_length, channels)
        x_seq = x_seq[:, self.left_pad:self.left_pad + length]
        return x_seq.transpose(1, 2)


class DilatedWindowAttentionClinVarCNN(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        layers = [nn.Conv1d(in_channels, 64, kernel_size=7, padding=3), nn.BatchNorm1d(64), nn.ReLU()]
        for dilation in [1, 2, 4, 8, 16, 32, 64]:
            layers.extend([
                nn.Conv1d(64, 64, kernel_size=7, padding=3 * dilation, dilation=dilation),
                nn.BatchNorm1d(64),
                nn.ReLU(),
                nn.Dropout(0.05),
            ])
        self.features = nn.Sequential(*layers)
        self.window_attention = WindowAttentionBlock1D(64, window_size=51, num_heads=4, dropout=0.10)
        self.classifier = nn.Sequential(nn.Linear(64 * 3, 96), nn.ReLU(), nn.Dropout(0.25), nn.Linear(96, 1))

    def forward(self, x):
        features = self.features(x)
        features = self.window_attention(features)
        center = features[:, :, features.shape[-1] // 2]
        global_max = torch.amax(features, dim=-1)
        global_mean = torch.mean(features, dim=-1)
        pooled = torch.cat([center, global_max, global_mean], dim=1)
        return self.classifier(pooled).squeeze(1)


def make_model():
    return DilatedWindowAttentionClinVarCNN(INPUT_CHANNELS)

model = make_model()
print(model.__class__.__name__)
print("parameters:", sum(p.numel() for p in model.parameters()))

In [ ]:
# =========================
# 12. Demo forward/backward test
# =========================
if RUN_DEMO:
    demo_indices = np.arange(min(8, len(X)))
    demo_loader = make_loader(demo_indices, positional_encoding, batch_size=8)
    demo_x, demo_y = next(iter(demo_loader))
    demo_model = make_model().to(DEVICE)
    demo_x = demo_x.to(DEVICE)
    demo_y = demo_y.to(DEVICE)
    demo_logits = demo_model(demo_x)
    assert demo_logits.shape == (len(demo_indices),)
    demo_loss = nn.BCEWithLogitsLoss()(demo_logits, demo_y)
    demo_loss.backward()
    print("forward/backward ok", "logits shape", tuple(demo_logits.shape), "loss", float(demo_loss.detach().cpu()))
    del demo_model, demo_x, demo_y, demo_logits, demo_loss
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# =========================
# 13. Training helpers and mini-training demo
# =========================
def safe_auc(y_true, y_score, kind):
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, y_score)) if kind == "roc" else float(average_precision_score(y_true, y_score))


def metrics_from_probs(y_true, y_prob, loss):
    y_pred = (y_prob >= 0.5).astype(int)
    return {"loss": float(loss), "accuracy": float(accuracy_score(y_true, y_pred)), "roc_auc": safe_auc(y_true, y_prob, "roc"), "pr_auc": safe_auc(y_true, y_prob, "pr")}


def make_scheduler(optimizer, total_steps, warmup_steps, min_lr_ratio=MIN_LR_RATIO):
    total_steps = max(1, total_steps)
    warmup_steps = max(0, min(warmup_steps, total_steps - 1))
    def lr_lambda(step):
        if warmup_steps > 0 and step < warmup_steps:
            return max((step + 1) / warmup_steps, min_lr_ratio)
        decay_steps = max(1, total_steps - warmup_steps)
        progress = min(1.0, max(0.0, (step - warmup_steps) / decay_steps))
        cosine = 0.5 * (1.0 + np.cos(np.pi * progress))
        return min_lr_ratio + (1.0 - min_lr_ratio) * cosine
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def run_epoch(model, loader, criterion, optimizer, device, label, scheduler=None, grad_clip_norm=None):
    train = optimizer is not None
    model.train(train)
    losses, all_probs, all_labels = [], [], []
    last_lr = float(optimizer.param_groups[0]["lr"]) if optimizer is not None else float("nan")
    last_grad_norm = float("nan")
    for x_batch, y_batch in tqdm(loader, desc=label, leave=False):
        x_batch = x_batch.to(device, non_blocking=True)
        y_batch = y_batch.to(device, non_blocking=True)
        if train:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(train):
            logits = model(x_batch)
            loss = criterion(logits, y_batch)
            if train:
                loss.backward()
                if grad_clip_norm is not None and grad_clip_norm > 0:
                    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)
                    last_grad_norm = float(grad_norm.detach().cpu())
                optimizer.step()
                if scheduler is not None:
                    scheduler.step()
                last_lr = float(optimizer.param_groups[0]["lr"])
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        labels = y_batch.detach().cpu().numpy()
        losses.append(float(loss.item()) * len(labels))
        all_probs.append(probs)
        all_labels.append(labels)
    y_true = np.concatenate(all_labels).astype(int)
    y_prob = np.concatenate(all_probs)
    metrics = metrics_from_probs(y_true, y_prob, float(np.sum(losses) / len(y_true)))
    if train:
        metrics["lr"] = last_lr
        metrics["grad_norm"] = last_grad_norm
    return metrics, y_prob, y_true


def run_eval_epoch(model, loader, rc_loader, criterion, device, label, rc_tta):
    metrics, probs, labels = run_epoch(model, loader, criterion, None, device, label)
    if not rc_tta:
        return metrics, probs, labels
    rc_metrics, rc_probs, rc_labels = run_epoch(model, rc_loader, criterion, None, device, f"{label} rc")
    assert np.array_equal(labels, rc_labels)
    tta_probs = (probs + rc_probs) / 2.0
    tta_loss = float((metrics["loss"] + rc_metrics["loss"]) / 2.0)
    return metrics_from_probs(labels, tta_probs, tta_loss), tta_probs, labels


def threshold_table(y_true, y_prob):
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    table = pd.DataFrame({"threshold": thresholds, "precision": precision[:-1], "recall": recall[:-1]})
    denom_f1 = table["precision"] + table["recall"]
    table["f1"] = np.where(denom_f1 > 0, 2 * table["precision"] * table["recall"] / denom_f1, 0.0)
    beta2 = 4.0
    denom_f2 = beta2 * table["precision"] + table["recall"]
    table["f2"] = np.where(denom_f2 > 0, (1.0 + beta2) * table["precision"] * table["recall"] / denom_f2, 0.0)
    return table


def metrics_at_threshold(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_pathogenic": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall_pathogenic": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1_pathogenic": float(f1_score(y_true, y_pred, zero_division=0)),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
    }


def train_one_split(split_mode, debug=False):
    train_idx, val_idx, test_idx, purged_val_idx, purged_test_idx, audit = prepare_split(split_mode, debug=debug)
    batch_size = DEBUG_BATCH_SIZE if debug else BATCH_SIZE
    epochs = 1 if debug else EPOCHS
    hard_negative_epochs = 0 if debug else HARD_NEGATIVE_EPOCHS
    train_loader = make_loader(train_idx, positional_encoding, batch_size, sampler=make_weighted_sampler(train_idx, y), random_reverse_complement=RC_AUGMENT)
    train_eval_loader = make_loader(train_idx, positional_encoding, batch_size)
    train_eval_rc_loader = make_loader(train_idx, positional_encoding, batch_size, reverse_complement=True) if RC_TTA else None
    val_loader = make_loader(val_idx, positional_encoding, batch_size)
    val_rc_loader = make_loader(val_idx, positional_encoding, batch_size, reverse_complement=True) if RC_TTA else None
    test_loader = make_loader(test_idx, positional_encoding, batch_size)
    test_rc_loader = make_loader(test_idx, positional_encoding, batch_size, reverse_complement=True) if RC_TTA else None

    labels_train = y[train_idx].astype(int)
    counts = np.bincount(labels_train, minlength=2)
    pos_weight_value = float(np.sqrt(counts[0] / max(counts[1], 1)))
    model = make_model().to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight_value, dtype=torch.float32, device=DEVICE))
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    total_steps = max(1, len(train_loader) * epochs)
    warmup_steps = int(round(WARMUP_EPOCHS * len(train_loader)))
    scheduler = make_scheduler(optimizer, total_steps, warmup_steps)
    history = []
    best_val_pr_auc = -np.inf
    best_state = None
    start_time = time.time()

    for epoch in range(1, epochs + 1):
        train_metrics, _, _ = run_epoch(model, train_loader, criterion, optimizer, DEVICE, f"{split_mode} train {epoch}", scheduler=scheduler, grad_clip_norm=GRAD_CLIP_NORM)
        val_metrics, _, _ = run_eval_epoch(model, val_loader, val_rc_loader, criterion, DEVICE, f"{split_mode} val {epoch}", RC_TTA)
        row = {"phase": "main", "epoch": epoch, "split_mode": split_mode, "scheduler": "cosine", "warmup_epochs": WARMUP_EPOCHS, "grad_clip_norm": GRAD_CLIP_NORM}
        row.update({f"train_{k}": v for k, v in train_metrics.items()})
        row.update({f"val_{k}": v for k, v in val_metrics.items()})
        history.append(row)
        print(row)
        if val_metrics["pr_auc"] > best_val_pr_auc:
            best_val_pr_auc = val_metrics["pr_auc"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    if hard_negative_epochs > 0:
        _, train_probs, train_labels = run_eval_epoch(model, train_eval_loader, train_eval_rc_loader, criterion, DEVICE, f"{split_mode} predict train", RC_TTA)
        train_order = train_idx
        pos_idx = train_order[train_labels == 1]
        hard_neg_idx = train_order[(train_labels == 0) & (train_probs >= 0.50)]
        easy_neg_idx = train_order[(train_labels == 0) & (train_probs < 0.50)]
        max_easy = int((len(pos_idx) + len(hard_neg_idx)) * 1.0)
        rng = np.random.default_rng(RANDOM_STATE)
        if len(easy_neg_idx) > max_easy:
            easy_neg_idx = rng.choice(easy_neg_idx, size=max_easy, replace=False)
        hard_train_idx = np.concatenate([pos_idx, hard_neg_idx, easy_neg_idx])
        rng.shuffle(hard_train_idx)
        hard_loader = make_loader(hard_train_idx, positional_encoding, batch_size, shuffle=True, random_reverse_complement=RC_AUGMENT)
        optimizer = torch.optim.AdamW(model.parameters(), lr=HARD_NEGATIVE_LR, weight_decay=WEIGHT_DECAY)
        for epoch in range(1, hard_negative_epochs + 1):
            train_metrics, _, _ = run_epoch(model, hard_loader, criterion, optimizer, DEVICE, f"{split_mode} hard train {epoch}", grad_clip_norm=GRAD_CLIP_NORM)
            val_metrics, _, _ = run_eval_epoch(model, val_loader, val_rc_loader, criterion, DEVICE, f"{split_mode} hard val {epoch}", RC_TTA)
            row = {"phase": "hard_negative", "epoch": epoch, "split_mode": split_mode, "scheduler": "none", "warmup_epochs": 0, "grad_clip_norm": GRAD_CLIP_NORM}
            row.update({f"train_{k}": v for k, v in train_metrics.items()})
            row.update({f"val_{k}": v for k, v in val_metrics.items()})
            history.append(row)
            print(row)
            if val_metrics["pr_auc"] > best_val_pr_auc:
                best_val_pr_auc = val_metrics["pr_auc"]
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        if best_state is not None:
            model.load_state_dict(best_state)

    final_val_metrics, final_val_probs, final_val_labels = run_eval_epoch(model, val_loader, val_rc_loader, criterion, DEVICE, f"{split_mode} final val", RC_TTA)
    test_metrics, test_probs, test_labels = run_eval_epoch(model, test_loader, test_rc_loader, criterion, DEVICE, f"{split_mode} test", RC_TTA)
    test_pred = (test_probs >= 0.5).astype(int)
    val_thresholds = threshold_table(final_val_labels, final_val_probs)
    test_thresholds = threshold_table(test_labels, test_probs)
    best_val_f1 = val_thresholds.loc[val_thresholds["f1"].idxmax()].to_dict()
    metrics = {
        "model_name": f"kaggle_cnn_positional_encoding_{CONTEXT_LENGTH}_dilated_window_attention_rcaug_rctta_{split_mode}" + (f"_purged{PURGE_DISTANCE_BP}" if split_mode == "purged_gene_group" else ""),
        "length": CONTEXT_LENGTH,
        "input_channels": INPUT_CHANNELS,
        "architecture": "dilated_window_attention",
        "split": split_mode,
        "split_audit": audit,
        "batch_size": batch_size,
        "epochs": epochs,
        "hard_negative_epochs": hard_negative_epochs,
        "pos_weight": pos_weight_value,
        "rc_augment": RC_AUGMENT,
        "rc_tta": RC_TTA,
        "final_val_metrics": final_val_metrics,
        "test_metrics": test_metrics,
        "test_precision_pathogenic_05": float(precision_score(test_labels, test_pred, zero_division=0)),
        "test_recall_pathogenic_05": float(recall_score(test_labels, test_pred, zero_division=0)),
        "test_f1_pathogenic_05": float(f1_score(test_labels, test_pred, zero_division=0)),
        "confusion_matrix": confusion_matrix(test_labels, test_pred).tolist(),
        "best_val_pr_auc": float(best_val_pr_auc),
        "best_val_f1_threshold": {k: float(v) for k, v in best_val_f1.items()},
        "test_at_best_val_f1_threshold": metrics_at_threshold(test_labels, test_probs, float(best_val_f1["threshold"])),
        "runtime_minutes": float((time.time() - start_time) / 60.0),
    }

    if not debug:
        safe_name = metrics["model_name"]
        torch.save({"model_state_dict": model.state_dict(), "config": metrics}, MODEL_DIR / f"{safe_name}_pytorch.pt")
        pd.DataFrame(history).to_parquet(PROCESSED_DIR / f"{safe_name}_training_history.parquet", index=False)
        test_thresholds.to_parquet(PROCESSED_DIR / f"{safe_name}_threshold_table.parquet", index=False)
        val_thresholds.to_parquet(PROCESSED_DIR / f"{safe_name}_val_threshold_table.parquet", index=False)
        pred_df = metadata_df.iloc[test_idx].copy()
        pred_df["y_true"] = test_labels
        pred_df["pred_proba_pathogenic"] = test_probs
        pred_df["pred_label"] = test_pred
        pred_df.to_parquet(PROCESSED_DIR / f"{safe_name}_test_predictions.parquet", index=False)
        np.savez_compressed(PROCESSED_DIR / f"{safe_name}_split_indices.npz", train_idx=train_idx, val_idx=val_idx, test_idx=test_idx, purged_val_idx=purged_val_idx, purged_test_idx=purged_test_idx)
        (PROCESSED_DIR / f"{safe_name}_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    return metrics

if RUN_DEMO:
    demo_metrics = train_one_split("purged_gene_group", debug=True)
    print(json.dumps({"demo_test_metrics": demo_metrics["test_metrics"], "demo_best_val_f1": demo_metrics["test_at_best_val_f1_threshold"]}, indent=2))

In [ ]:
# =========================
# 14. Full train/eval loop
# =========================
metrics_results = []
if RUN_FULL_TRAIN:
    for split_mode in SPLIT_MODES:
        safe_name = f"kaggle_cnn_positional_encoding_{CONTEXT_LENGTH}_dilated_window_attention_rcaug_rctta_{split_mode}" + (f"_purged{PURGE_DISTANCE_BP}" if split_mode == "purged_gene_group" else "")
        metrics_path = PROCESSED_DIR / f"{safe_name}_metrics.json"
        if metrics_path.exists() and not FORCE_RETRAIN:
            print("Using existing metrics:", metrics_path)
            metrics = json.loads(metrics_path.read_text())
        else:
            metrics = train_one_split(split_mode, debug=False)
        metrics_results.append(metrics)
else:
    for path in PROCESSED_DIR.glob(f"kaggle_cnn_positional_encoding_{CONTEXT_LENGTH}_dilated_window_attention_rcaug_rctta_*_metrics.json"):
        metrics_results.append(json.loads(path.read_text()))

print("Finished runs:", [m["model_name"] for m in metrics_results])

In [ ]:
# =========================
# 15. Comparison table
# =========================
rows = []
for m in metrics_results:
    split_audit = m.get("split_audit", {})
    rows.append({
        "model_name": m["model_name"],
        "split": m["split"],
        "length": m["length"],
        "test_roc_auc": m["test_metrics"]["roc_auc"],
        "test_pr_auc": m["test_metrics"]["pr_auc"],
        "test_accuracy": m["test_metrics"]["accuracy"],
        "precision@0.5": m["test_precision_pathogenic_05"],
        "recall@0.5": m["test_recall_pathogenic_05"],
        "f1@0.5": m["test_f1_pathogenic_05"],
        "best_val_f1_threshold": m["best_val_f1_threshold"]["threshold"],
        "test_f1_at_best_val_f1": m["test_at_best_val_f1_threshold"]["f1_pathogenic"],
        "gene_overlap_train_test": split_audit.get("gene_overlap_train_test"),
        "test_min_distance_bp": split_audit.get("test_nearest_train_val_distance_bp", {}).get("min"),
        "runtime_minutes": m.get("runtime_minutes"),
    })
comparison_df = pd.DataFrame(rows).sort_values(["split", "test_pr_auc"], ascending=[True, False])
display(comparison_df)

preferred = comparison_df[comparison_df["split"].eq("purged_gene_group")]
if len(preferred):
    print("Preferred strict benchmark:")
    display(preferred.sort_values("test_pr_auc", ascending=False).head(1))